In [0]:
%sql
-- ============================================================================
-- 7. mkt_dashboard_top_films_monthly  (grain: month x rank, 5 rows/month)
--    Fills every closed month after the table's last report_month.
--    Transactions only, so a month is ready once it has ended (up to yesterday).
--    Query = your top films backfill, unchanged; only the loop changed
--    (no TRUNCATE, no fixed 24-month range).
-- ============================================================================
BEGIN
  DECLARE v_month_start DATE;
  DECLARE v_month_end   DATE;
  DECLARE v_end         DATE;

  SET v_month_start = (
    SELECT add_months(MAX(report_month), 1)
    FROM data_exploration_hub.marketing_hub.mkt_dashboard_top_films_monthly
  );
  SET v_end = date_sub(current_date(), 1);

  WHILE last_day(v_month_start) <= v_end DO
    SET v_month_end = last_day(v_month_start);

    DELETE FROM data_exploration_hub.marketing_hub.mkt_dashboard_top_films_monthly
    WHERE report_month = v_month_start;

        INSERT INTO data_exploration_hub.marketing_hub.mkt_dashboard_top_films_monthly
        WITH
        ticket_lines AS (
            SELECT
                t.film_id,
                t.Quantity,
                CASE WHEN t.CineClubFLAG = 'Yes' THEN 1 ELSE 0 END AS is_cc,
                CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes' THEN 1 ELSE 0 END AS is_scene
            FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
            JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc
                ON t.LocationId = loc.LocationID
            WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
              AND t.TicketFLAG = 'Yes'
              AND t.VisitDateId BETWEEN CAST(DATE_FORMAT(v_month_start, 'yyyyMMdd') AS INT)
                                     AND CAST(DATE_FORMAT(v_month_end, 'yyyyMMdd') AS INT)
        ),

        film_attendance AS (
            SELECT
                f.title_name,
                SUM(CASE WHEN tl.is_cc = 1 THEN tl.Quantity ELSE 0 END)       AS cc_tickets,
                SUM(CASE WHEN tl.is_scene = 1 THEN tl.Quantity ELSE 0 END)    AS scene_tickets,
                SUM(tl.Quantity)                                              AS total_biz_tickets
            FROM ticket_lines tl
            JOIN cpx_dataanalytics_gold.customer360.cpx_d_film f
                ON tl.film_id = f.film_id
            GROUP BY f.title_name
        ),

        ranked AS (
            SELECT
                title_name,
                cc_tickets,
                scene_tickets,
                total_biz_tickets,
                ROW_NUMBER() OVER (ORDER BY cc_tickets DESC) AS rnk
            FROM film_attendance
        )

        SELECT
            v_month_start   AS report_month,
            r.rnk           AS rank,
            r.title_name,
            r.cc_tickets,
            r.scene_tickets,
            r.total_biz_tickets
        FROM ranked r
        WHERE r.rnk <= 5
        ORDER BY r.rnk;

    SET v_month_start = add_months(v_month_start, 1);
  END WHILE;
END;